# softmax from logits — procedural drill

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `softmax-from-logits`. When a test cell passes, your progress is reported back to your account.

**What you'll practice.** Five softmax patterns that ramp from naive 1-D softmax → subtract-max stable softmax → row-wise softmax → log-softmax via logsumexp → stable per-sample cross-entropy. Read the docstring, fill the function body, run the test cell. The solution sits in the collapsed `<details>` block below each exercise.

**Per-exercise structure** (Doughty et al. ACE 2024 — `[Bloom level] + [LO] + [Keywords] + [KCs]`):
Each exercise begins with a yaml block stating its Bloom cognitive level, learning objective, keywords, and the knowledge components (KCs) it targets. This makes the cognitive demand explicit instead of buried.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import torch.nn.functional as F

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Applied patterns and advanced` subtopic.
You can copy the token from your Delta Drills account page.

This drill exercises the **atom `softmax-from-logits`**, which bridges to the bank subtopic `Numpy: Applied patterns and advanced` for EWMA state. Completing all 5 exercises triggers a single `arena-rating` beacon at the end of the notebook.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "softmax-from-logits"
DD_SUBTOPIC = "Numpy: Applied patterns and advanced"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

# Track which exercises passed in this session.
_dd_passed = set()

## Softmax — quick refresher

**Naive:** `softmax(x)_i = exp(x_i) / sum_j exp(x_j)`. Overflows whenever any `x_i` is large.

**Stable:** `softmax(x) = softmax(x - max(x))`. Same values, but every exponent is ≤ 0 so `exp` stays in `(0, 1]`.

**Log-softmax:** `log_softmax(x) = x - logsumexp(x)`. Avoids materializing `exp(x)` so it stays precise for tiny probabilities.

**Cross-entropy from logits:** `loss_n = -log_softmax(logits_n)[target_n]`. The canonical classification objective — never go through `softmax` first.

### Exercise 1 — naive softmax (1-D logits)

> ```yaml
> Difficulty: ⚪⚪⚪⚪⚪
> Bloom level: Remember
> LO: Recall the softmax formula `exp(x) / sum(exp(x))` and apply it to a 1-D tensor.
> Keywords: softmax, exp, naive-formula
> ```

**KCs targeted:** `softmax-naive-formula`

Implement `ex1_softmax_naive(logits)` using the literal softmax formula:

`softmax(x)_i = exp(x_i) / sum_j exp(x_j)`.

Input: 1-D tensor of small values (so overflow doesn't matter yet). Output: same shape, every entry in [0, 1], sums to 1.

Don't use `torch.softmax` — write the formula directly. We'll deal with overflow in the next exercise.

In [ ]:
def ex1_softmax_naive(logits: Tensor) -> Tensor:
    """Naive 1-D softmax. exp(x) / sum(exp(x))."""
    raise NotImplementedError()


def _test_ex1():
    logits = t.tensor([1.0, 2.0, 3.0, -1.0])
    p = ex1_softmax_naive(logits)
    assert p.shape == logits.shape, f'shape mismatch: {p.shape}'
    assert (p >= 0).all() and (p <= 1).all(), 'probabilities must lie in [0, 1]'
    assert t.allclose(p.sum(), t.tensor(1.0)), f'probabilities should sum to 1, got {p.sum().item()}'
    assert t.allclose(p, t.softmax(logits, dim=0)), 'values differ from t.softmax'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_softmax_naive(logits: Tensor) -> Tensor:
    e = logits.exp()
    return e / e.sum()
```
</details>

### Exercise 2 — stable softmax (subtract-max trick)

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Bloom level: Apply
> LO: Apply the subtract-max trick to make softmax numerically stable for large logits.
> Keywords: stable-softmax, subtract-max, overflow-guard
> ```

**KCs targeted:** `softmax-subtract-max`

Implement `ex2_softmax_stable(logits)` so that even logits of magnitude 1000 don't produce `inf` or `nan`.

Subtract the maximum logit before exponentiating. Mathematically a no-op (softmax is shift-invariant), but it keeps every exponent ≤ 0, so `exp` stays in `(0, 1]` instead of overflowing.

Input: 1-D tensor. Output: same shape, sums to 1, no `inf`/`nan` even for extreme inputs.

In [ ]:
def ex2_softmax_stable(logits: Tensor) -> Tensor:
    """Stable 1-D softmax. exp(x - max(x)) / sum(exp(x - max(x)))."""
    raise NotImplementedError()


def _test_ex2():
    # Normal case still correct
    logits = t.tensor([1.0, 2.0, 3.0, -1.0])
    p = ex2_softmax_stable(logits)
    assert t.allclose(p, t.softmax(logits, dim=0)), 'normal-case mismatch'

    # Overflow stress test — naive softmax would NaN here.
    huge = t.tensor([1000.0, 1001.0, 999.0])
    ph = ex2_softmax_stable(huge)
    assert not t.isnan(ph).any() and not t.isinf(ph).any(), 'output contains NaN/Inf — subtract-max not applied'
    assert t.allclose(ph.sum(), t.tensor(1.0)), f'sum should be 1, got {ph.sum().item()}'
    assert t.allclose(ph, t.softmax(huge, dim=0)), 'overflow-case mismatch'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_softmax_stable(logits: Tensor) -> Tensor:
    shifted = logits - logits.max()
    e = shifted.exp()
    return e / e.sum()
```

**Why this works.** `softmax(x + c) = softmax(x)` for any constant `c`. Setting `c = -max(x)` shifts the largest input to 0 — its `exp` becomes 1 (the maximum possible non-overflowing value), and every other `exp` is in `(0, 1)`. No overflow, no precision loss in the dominant term.
</details>

### Exercise 3 — row-wise softmax (axis-aware with keepdim)

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply softmax to each row of a 2-D logits tensor using `keepdim=True` for the max and the sum.
> Keywords: axis-aware, keepdim, batched-softmax
> ```

**KCs targeted:** `softmax-axis-aware`

Implement `ex3_softmax_rows(logits)` to apply softmax independently to every row of a 2-D tensor.

Input shape: `(N, C)` — batch of `N` rows, each over `C` classes. Output shape: `(N, C)`. Every row of the output should be a probability distribution (sums to 1).

Use the subtract-max + keepdim pattern. `max(dim=1, keepdim=True)` returns a NamedTuple `(values, indices)` — you want `.values`. Same for the sum: keep its axis with `keepdim=True` so it broadcasts back.

In [ ]:
def ex3_softmax_rows(logits: Tensor) -> Tensor:
    """Row-wise stable softmax. (N, C) → (N, C), each row sums to 1."""
    raise NotImplementedError()


def _test_ex3():
    logits = t.tensor([
        [1.0, 2.0, 3.0],
        [-2.0, 0.0, 1.0],
        [1000.0, 999.0, 1001.0],   # overflow stress
    ])
    p = ex3_softmax_rows(logits)
    assert p.shape == logits.shape, f'shape mismatch: {p.shape}'
    row_sums = p.sum(dim=1)
    assert t.allclose(row_sums, t.ones(3)), f'each row should sum to 1, got {row_sums.tolist()}'
    assert not t.isnan(p).any() and not t.isinf(p).any(), 'NaN/Inf in output — subtract-max missing'
    assert t.allclose(p, t.softmax(logits, dim=1)), 'values differ from t.softmax(logits, dim=1)'
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
def ex3_softmax_rows(logits: Tensor) -> Tensor:
    row_max = logits.max(dim=1, keepdim=True).values
    e = (logits - row_max).exp()
    return e / e.sum(dim=1, keepdim=True)
```

**`keepdim=True` is critical twice here.** Without it on `max`, you'd get a 1-D row of per-batch maxes that wouldn't broadcast against the 2-D logits. Same for the sum in the divisor. With `keepdim=True` both stay 2-D `(N, 1)` and broadcast cleanly against `(N, C)`.
</details>

### Exercise 4 — log-softmax via logsumexp (no exp materialized)

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the identity `log_softmax(x) = x - logsumexp(x)` to compute log-probabilities directly.
> Keywords: log-softmax, logsumexp, cross-entropy-prep
> ```

**KCs targeted:** `softmax-logsumexp`

Implement `ex4_log_softmax_rows(logits)` to return `log(softmax(logits))` row-wise.

Input shape: `(N, C)`. Output shape: `(N, C)`. Every row's `exp` of the output should sum to 1 (i.e. they're valid log-probabilities).

Don't compute softmax and then take the log — that loses precision for near-zero probabilities. Use the identity:

`log_softmax(x) = x - logsumexp(x)`

where `logsumexp(x) = max(x) + log(sum(exp(x - max(x))))`. PyTorch ships `torch.logsumexp(x, dim=..., keepdim=True)` — use it.

Equivalent to `torch.log_softmax(logits, dim=1)`.

In [ ]:
def ex4_log_softmax_rows(logits: Tensor) -> Tensor:
    """Log-softmax along dim=1. (N, C) → (N, C)."""
    raise NotImplementedError()


def _test_ex4():
    logits = t.tensor([
        [1.0, 2.0, 3.0, -1.0],
        [1000.0, 1001.0, 1002.0, 999.0],
    ])
    lp = ex4_log_softmax_rows(logits)
    assert lp.shape == logits.shape, f'shape mismatch: {lp.shape}'
    # exp of log-probs should sum to 1 per row.
    assert t.allclose(lp.exp().sum(dim=1), t.ones(2), atol=1e-5), 'rows of exp(log_softmax) should sum to 1'
    assert not t.isnan(lp).any() and not t.isinf(lp).any(), 'NaN/Inf in output'
    assert t.allclose(lp, t.log_softmax(logits, dim=1), atol=1e-5), 'values differ from t.log_softmax'
    _dd_passed.add('ex4')
    print("ex4 ✓")

_test_ex4()

<details><summary>Solution</summary>

```python
def ex4_log_softmax_rows(logits: Tensor) -> Tensor:
    return logits - t.logsumexp(logits, dim=1, keepdim=True)
```

**Why not `softmax(x).log()`?** When the true probability of class i is 1e-30, `softmax(x)[i]` underflows to 0 and `log(0) = -inf`. Computing `log_softmax` directly avoids the underflow because the subtraction stays in the log domain.

**Why `logsumexp` is stable.** It's secretly the subtract-max trick again: `logsumexp(x) = max(x) + log(sum(exp(x - max(x))))`. The exp inside is on shifted-down values so it never overflows.
</details>

### Exercise 5 — stable per-sample cross-entropy from logits

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Synthesize stable log-softmax + class-index lookup to compute per-sample cross-entropy directly from raw logits.
> Keywords: cross-entropy, logsumexp, integration, multi-kc
> ```

**KCs targeted:** `softmax-subtract-max`, `softmax-logsumexp`, `softmax-cross-entropy-stable`

Implement `ex5_cross_entropy_per_sample(logits, targets)` to compute per-sample cross-entropy from raw logits and integer class targets.

Input shapes: `logits` is `(N, C)`, `targets` is `(N,)` of class indices in `[0, C)`. Output shape: `(N,)` — one loss per sample.

The math: `loss_n = -log_softmax(logits_n)[targets_n]`. Use the logsumexp identity from Exercise 4 — never materialize `exp` or intermediate probabilities.

Two well-known ways to do the class-index lookup:
- `logits.gather(1, targets.unsqueeze(1)).squeeze(1)`
- `logits[torch.arange(N), targets]` (integer-array indexing)

Equivalent to `torch.nn.functional.cross_entropy(logits, targets, reduction='none')`.

> ⚠️ **Integrative exercise.** This combines 3+ KCs (subtract-max, logsumexp, integer indexing) in one expression; empirical work (Lohr et al. ITiCSE 2025) shows 3-concept LLM-generated exercises drop from ~94% to ~40% solvability. Expect a step in difficulty here vs Exercises 1-4.

In [ ]:
def ex5_cross_entropy_per_sample(logits: Tensor, targets: Tensor) -> Tensor:
    """Per-sample cross-entropy from raw logits. (N, C), (N,) → (N,)."""
    raise NotImplementedError()


def _test_ex5():
    logits = t.tensor([
        [1.0, 2.0, 3.0, -1.0],
        [-2.0, 0.0, 1.0, 0.5],
        [1000.0, 1001.0, 999.0, 998.0],   # overflow stress
    ])
    targets = t.tensor([2, 1, 0])
    loss = ex5_cross_entropy_per_sample(logits, targets)
    assert loss.shape == (3,), f'expected (3,), got {tuple(loss.shape)}'
    assert not t.isnan(loss).any() and not t.isinf(loss).any(), 'NaN/Inf in output'
    expected = F.cross_entropy(logits, targets, reduction='none')
    assert t.allclose(loss, expected, atol=1e-5), f'values differ from F.cross_entropy:\n  got      {loss}\n  expected {expected}'
    _dd_passed.add('ex5')
    print("ex5 ✓")

_test_ex5()

<details><summary>Solution</summary>

```python
def ex5_cross_entropy_per_sample(logits: Tensor, targets: Tensor) -> Tensor:
    log_probs = logits - t.logsumexp(logits, dim=1, keepdim=True)
    N = logits.shape[0]
    return -log_probs[t.arange(N), targets]
```

**Reading the pattern.**
- `logits - logsumexp(logits, dim=1, keepdim=True)` → log-probabilities, stable for any logit magnitude.
- `log_probs[arange(N), targets]` → integer-array indexing picks `log_probs[n, targets[n]]` for every `n`.
- Negate → cross-entropy (we minimize `-log p(correct class)`).

**Equivalent gather form:** `-log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)`. Functionally identical; `gather` is preferred when `targets` has more dims (e.g. language-modeling next-token prediction with shape `(B, T)`).
</details>

## Done

Run the cell below to report your progress to Delta Drills. The beacon fires only if all 5 exercises passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1', 'ex2', 'ex3', 'ex4', 'ex5'}

def _dd_feedback_level(num_passed: int) -> str:
    """Map exercise-pass count → arena-rating feedback enum."""
    if num_passed == 5: return 'not_much'
    if num_passed >= 3: return 'somewhat'
    return 'a_lot'

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {len(missing)} exercises still failing: {sorted(missing)}.")
        print("[Delta Drills] not reporting until all 5 pass.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}',
        'subtopics': [DD_SUBTOPIC],
        'feedback': _dd_feedback_level(len(_dd_passed)),
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()